# So sánh mô hình (feature 010) — Colab T4

Chạy các mô hình GGUF qua **đúng code serving** (prompt, llama.cpp, chặn lặp, ghép JSON) trên các tập đánh giá, ghi dự đoán lên Drive. Claude chấm điểm bằng eval harness (biến thể B) trên máy.

| Mô hình | File | Prompt |
|---|---|---|
| `base` | Qwen3-1.7B gốc, chưa fine-tune (Hugging Face `unsloth/Qwen3-1.7B-GGUF`, Q4_K_M) | v3 (zero-shot) |
| `v1`, `v2`, `v3` | GGUF đã fine-tune trên Drive dự án | v1 / v2 / v3 |

Tập dữ liệu (thư mục `MyDrive/cv-analysis-service/retrain_v3/` đã upload khi train v3): `val` (99), `val_swap` (99), `dev` (45 CV sinh theo bố cục thật), `test_real` (20 CV thật).

**Cách chạy**: Runtime → T4 GPU → Run all. Mặc định (`FULL=False`) mất ~1–1,5 giờ: `base` trên cả 4 tập, `v1`/`v2`/`v3` trên `dev` + `test_real` (các tập còn lại đã có số liệu). `FULL=True` chạy mọi mô hình trên mọi tập (~2,5–3 giờ). Ngắt giữa chừng thì Run all lại: phần đã chạy được giữ trên Drive.

Xong thì báo Claude; kết quả ở `MyDrive/cv-analysis-service/eval_models/`.


In [ ]:
# [1] Cấu hình
FULL = False
REPO, BRANCH = "https://github.com/long20102004/recruit_system.git", "hocnv/DO-06"
DRIVE_ROOT = "/content/drive/MyDrive/cv-analysis-service"
DATA_DIR = f"{DRIVE_ROOT}/retrain_v3"
OUT_DIR = f"{DRIVE_ROOT}/eval_models"
MODELS = {   # name: (nguồn, prompt version)
    "base": ("hf:https://huggingface.co/unsloth/Qwen3-1.7B-GGUF/resolve/main/Qwen3-1.7B-Q4_K_M.gguf", "v3"),
    "v1":   ("drive:1Kt4BbL22tR1HLw16f2tyGyus6mQDI2SV", "v1"),
    "v2":   ("drive:1X4A2sVFDWbtIIP6hwjjJuiA3Sb-mjrUY", "v2"),
    "v3":   ("drive:1mqmolAF-4nJXg1db89oHAYZKYlZ3CUcq", "v3"),
}
ALL_SPLITS = ["test_real", "dev", "val", "val_swap"]
PLAN = {m: ALL_SPLITS for m in MODELS} if FULL else {"base": ALL_SPLITS, "v1": ["dev", "test_real"], "v2": ["dev", "test_real"], "v3": ["dev", "test_real"]}
print(PLAN)


In [ ]:
# [2] Code đúng branch + llama.cpp CUDA
%cd /content
!rm -rf /content/recruit_system
!git clone -q --depth 1 -b {BRANCH} {REPO} /content/recruit_system
%cd /content/recruit_system/ai-service
!git log --oneline -1
!pip install -q fastapi uvicorn pydantic json-repair rapidfuzz gdown
!pip install -q nvidia-cuda-runtime-cu12 nvidia-cublas-cu12
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
import glob, site, os
lib_dirs = sorted({d for sp in site.getsitepackages() for d in glob.glob(os.path.join(sp, "nvidia", "*", "lib"))})
open("/etc/ld.so.conf.d/nvidia-pip.conf", "w").write("\n".join(lib_dirs + ["/usr/local/cuda/lib64"]) + "\n")
!ldconfig
os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
!python -c "import llama_cpp; print('GPU offload:', llama_cpp.llama_supports_gpu_offload())"


In [ ]:
# [3] Drive + tải model
from google.colab import drive
drive.mount("/content/drive")
import subprocess
os.makedirs("/content/models", exist_ok=True)
for split in ALL_SPLITS:
    assert os.path.exists(f"{DATA_DIR}/{split}.jsonl"), f"Thiếu {DATA_DIR}/{split}.jsonl"
paths = {}
for name, (src, _) in MODELS.items():
    if not PLAN.get(name):
        continue
    path = f"/content/models/{name}.gguf"
    if not os.path.exists(path):
        kind, ref = src.split(":", 1)
        cmd = ["gdown", "-q", ref, "-O", path] if kind == "drive" else ["wget", "-q", "-O", path, ref]
        subprocess.run(cmd, check=True)
    assert open(path, "rb").read(4) == b"GGUF", f"{name}: tải lỗi, xoá file và chạy lại"
    paths[name] = path
    print(name, round(os.path.getsize(path) / 1e9, 2), "GB")


In [ ]:
# [4] Chạy dự đoán (đúng code serving), ghi lên Drive — chạy lại được, bỏ qua CV đã có
%cd /content/recruit_system/ai-service
for name, splits in PLAN.items():
    args = ["python", "-m", "eval.run_models", "--gguf", paths[name], "--prompt-version", MODELS[name][1],
            "--out", f"{OUT_DIR}/{name}", "--n-gpu-layers", "-1"]
    for split in splits:
        args += ["--split", f"{split}={DATA_DIR}/{split}.jsonl"]
    print("=" * 20, name, "=" * 20, flush=True)
    subprocess.run(args, check=True)
print("Xong. Kết quả:", OUT_DIR)


In [ ]:
# [5] Tóm tắt (chỉ số đếm)
import json
for name in PLAN:
    for split in PLAN[name]:
        p = f"{OUT_DIR}/{name}/{split}/predictions.jsonl"
        if os.path.exists(p):
            rows = [json.loads(l) for l in open(p)]
            ok = sum(r["status"] == 200 for r in rows)
            cut = sum(bool(r.get("truncated")) for r in rows)
            lat = sum(r["latency_s"] for r in rows) / max(1, len(rows))
            print(f"{name:5} {split:9} {len(rows):3} CV | JSON hợp lệ {ok:3} | bị cắt {cut:2} | {lat:5.1f} s/CV")
